# DIP-STER Experiments Juypter Notebook 

In this notebook, we show how to prepare a tiltseries and run a network with DIP-STER

## 1. Imports and initialization


In [ ]:

## standard library imports
import os
import logging
import dataclasses

## Data processing imports
import torch
import numpy as np

#Image processing imports 
import stackview
import tomobase
tomobase.logger.setLevel(logging.INFO)
from tomobase.core.data_classes import Sinogram, Volume

import dipster as dip

tomobase.bootstrap(qt_enabled=True)

@dataclasses.dataclass
class TorchEnabledSinogram():
    data: torch.Tensor
    times: torch.Tensor
    angles: torch.Tensor


In [ ]:
#### User Input ####

## Select the root data folder containing all subsequent directories
main_dir =  #'/path/to/main/directory'

tiltseries_dir = 'TiltSeries128'  # Tilt Series folder 
model_dir = 'Model128'  # Trained DIP-STER Models folder
reconstruction_dir = 'DIP128'  # DIP-STER Reconstruction folder

tiltseries_name = 'Simu_Nanoshell'  # Name of the tilt series (without file extension)
tiltseries_file = tiltseries_name + '.mrc'  # File containing the tilt series

## Definition of all the necessary paths
path_tiltseries = os.path.join(main_dir, tiltseries_dir, tiltseries_file)
directory_model = os.path.join(main_dir, model_dir)



## 2. Tiltseries Processing

Prior to reconstruction the tiltseries must be sorted by time, normalized and have the orientation converted to (x, y, n). The final tiltseries is sent to the GPU using Pytorch

In [ ]:
tiltseries = Sinogram.read(path_tiltseries)
tiltseries.sort(by='times')

max_val, min_val = tiltseries.data.max(), tiltseries.data.min()
tiltseries.data = (tiltseries.data - min_val) / (max_val - min_val)

stackview.slice(tiltseries.values) # check the loaded tilt series is rotating around the vertical axis

In [ ]:

#Convert to Torch Structured Tensor
torch_tiltseries  = TorchEnabledSinogram(
    data = torch.from_numpy(np.transpose(tiltseries.values, (2,1,0))).to("cuda"),
    times = torch.from_numpy(tiltseries.times).to("cuda"),
    angles = torch.from_numpy(tiltseries.angles).to("cuda")
)


## 3. Reconstruction

To reconstruct, input the tilt series 

In [ ]:
net = dip.Solver(torch_tiltseries)

net.params.wandb_local_dir = "wandb"
net.params.wandb_project = 'Example_Project' 
net.params.noise_regularizer = 0

epochs = 20
net.params.max_steps=torch_tiltseries.data.shape[0] * torch_tiltseries.data.shape[2] * epochs # should be 10-20 epochs
net.params.style_size=16
net.params.up_factor = 8
net.params.depth = 1
net.params.hidden_dim = 512
net.params.batch_size = 8
net.params.gamma = 0.985
net.params.lr = 0.0001
net.params.step_size = int(0.03125 * net.params.max_steps)

print( f'Epochs: {epochs}, Max Steps: {net.params.max_steps}, Step Size: {net.params.step_size}')
meta = net.params.to_dict()

net.eval()
net.train(torch_tiltseries)

model_name = tiltseries_name +'-'+ net.params.wandb_name
path_model = os.path.join(directory_model, model_name +'dip.pkl')

torch.save(net.state_dict(), path_model)




In [ ]:
newnet = dip.Solver.from_state_dict(torch.load(path_model, map_location=torch.device('cpu'))) #load old models from saved pkls